# Building agent loop: Client SDK

Tool definition

In [3]:
import json

TOOLS = [
    {
        "name": "get_order_status",
        "description": (
            "Get the status of an order by providing the order ID."
            "Use this when customers ask about their order, shipment or delivery."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The unique identifier of the order.",
                }
            },
            "required": ["order_id"],
        },
    },
    {
        "name": "cancel_order",
        "description": (
            "Cancel order that has not yet shipped. "
            "Use this only when the customer explicitly requests cancellation."
        ),
        "input_schema": {
            "type" : "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The unique identifier of the order to be canceled.",
                },
            },
            "required": ["order_id"]    
        }
    }
]

def get_order_status(order_id: str) -> str: 
    mock_orders = {
        "5678": "Your order #5678 is currently being processed and will be shipped soon.",
        "1234": "Your order #1234 has been shipped and is expected to arrive",
        "9999": "Processing — not yet dispatched."
    }
    return mock_orders.get(order_id, f"Order ID {order_id} not found. Please check the order ID and try again.")

def cancel_order(order_id: str) -> str:
    cancellable = {"9999"}
    already_shipped = {"1234"}

    if order_id in cancellable:
        return f"Order ID {order_id} has been successfully canceled."
    if order_id in already_shipped:
        return (f"ERROR: Order {order_id} has already been shipped and cannot be canceled."
                "Customer may request a return after delivery.")
    return f"Order ID {order_id} not found. Please check the order ID and try again."

Writing the agent loop

In [4]:
import anthropic
import os
from dotenv import load_dotenv
load_dotenv()

model = "claude-sonnet-5-5"
client = anthropic.Anthropic()

SYSTEM_PROMPT = "You are a customer support agent. Help customers with order enqueries."

def run_agent(task: str) -> str:
    messages = [{"role": "user", "content": task}]

    while True:
        print(f"messages.count: {len(messages)}")
        response = client.messages.create(
            model = model,
            max_tokens=1000,
            system = SYSTEM_PROMPT,
            tools = TOOLS,
            messages = messages
        )
        print(response.stop_reason)
        if response.stop_reason == "end_turn":
            for block in response.content:
                if block.type == "text":
                    return block.text

        if response.stop_reason == "tool_use":
            # append the assistant's tool use request to the messages list
            messages.append({"role": "assistant", "content": response.content})

            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = dispatch_tool(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })
            messages.append({"role": "user", "content": tool_results})
            continue
        if response.stop_reason == "max_tokens":
            raise RuntimeError(
                f"Response truncated after {response.usage.output_tokens} tokens. "
                "Increase max_tokens or shorten the task."
            )

        if response.stop_reason == "stop_sequence":
            for block in response.content:
                if block.type == "text":
                    return block.text
                else:
                    return ""

def dispatch_tool(tool_name: str, tool_input: dict) -> str:
    if tool_name == "get_order_status":
        order_id = tool_input.get("order_id")
        return get_order_status(order_id)
    elif tool_name == "cancel_order":
        order_id = tool_input.get("order_id")
        return cancel_order(order_id)
    else:
        return f"Tool {tool_name} not recognized."

Running the agent

In [2]:
print("--- Success path ---")
answer = run_agent("Please cancel my order #9999")
print("Agent's response:", answer)


#print("--- Error path ---")
#answer = run_agent("Please cancel my order #1234?")
#print("Agent's response:", answer)

--- Success path ---
messages.count: 1


NameError: name 'TOOLS' is not defined